In [265]:
!pip install pyserial

In [266]:
import serial, time
!pip install pyserial

In [295]:
ser.close()

**Note:** if importing `serial` causes an error, you need to install the `pyserial` module using `pip`:

`pip install pyserial`

or 

`pip3 install pyserial`

Windows users should use the Anaconda prompt.  Mac users should be able to use the terminal.

In [268]:
print(serial)

<module 'serial' from 'C:\\Users\\caleb\\anaconda3\\Lib\\site-packages\\serial\\__init__.py'>


In [269]:
print(serial.__file__)

C:\Users\caleb\anaconda3\Lib\site-packages\serial\__init__.py


In [270]:
print(serial.__version__)

3.5


In [271]:
serial.VERSION

'3.5'

**Note:** if you serial version is 2.x, we might need to make changes to the code below

In [272]:
baudrate = 115200

In [273]:
#portname = '/dev/cu.usbmodem11301'#mac
portname = 'COM4'#windows

In [274]:
#ser.close()

In [275]:
ser = serial.Serial(portname, baudrate, timeout=5)

In [276]:
ser.in_waiting

0

In [277]:
#ser.close()

In [278]:
#read_all(ser)

In [279]:
def read_all(ser):
    out = []
    while ser.in_waiting > 0:
        data1b = ser.read(1)
        data1 = data1b.decode('utf-8')
        out.append(data1)
        
    outstr = ''.join(out)
    return outstr

In [280]:
read_all(ser)

'dual servo control over serial\n'

In [281]:
def read_one_line(ser):
    out = []
    while ser.in_waiting > 0:
        data1b = ser.read(1)
        data1 = data1b.decode('utf-8')
        if data1 in ['\n','\r']:
            break
        out.append(data1)
        
    outstr = ''.join(out)
    return outstr

In [282]:
read_one_line(ser)

''

In [283]:
read_all(ser)

''

In [284]:
def one_byte_int_to_serial_byte(int_byte):
    out_byte = int(int_byte).to_bytes(1, byteorder='big')
    return out_byte

In [285]:
def WriteByte(ser, bytein):
    out_byte = one_byte_int_to_serial_byte(bytein)
    ser.write(out_byte)

## Example

In [286]:
#byte1 = 7
#WriteByte(ser,byte1)#<--
#time.sleep(0.1)
#byte2 = 156
#WriteByte(ser,byte2)#<--
#time.sleep(0.1)
#next_line = read_one_line(ser)
#extra = read_all(ser)
#print('next_line: %s' % next_line)
#print('extra: %s' % extra)

# Break an integer into two bytes

In [287]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from numpy import sin, cos, tan, pi
import robotics
from robotics import Rx, Ry, Rz, sind, cosd, DH, prettymat
rtd = 180/pi
dtr = pi/180

In [288]:
# inputs from user
xll = 10 # x origin
yll = 33 # y origin
w =  -6  # width
h = 6   # height
#N = 2
K = 6  # number of steps per side
N = (2*K) #refining the robots movement

In [289]:
#define step size
dx = w/N
dy = h/N

#generate bottom coordinants
x_bottom = np.linspace(xll, xll+w-dx, N)  
y_bottom = np.full(N, yll)

#generate right coordinants
x_right = np.full(N, xll + w)  
y_right = np.linspace(yll, yll+h-dy, N)

#generate top coordinants
x_top = np.linspace(xll+w, xll+dx, N)  
y_top = np.full(N,yll+h)

#generate left coordinants
x_left = np.full(N+2, xll)  
y_left = np.linspace(yll+h, yll, N+1)
y_left = np.append(y_left, yll)
#print(x_left)
#print(y_left)
#combine bottom, right, top, left into one array
x_path = np.concatenate((x_bottom, x_right, x_top, x_left), axis=0) 
y_path = np.concatenate((y_bottom, y_right, y_top, y_left), axis=0) 

#combine x and y into one array
tip_path = np.column_stack((x_path, y_path))
tip_path

array([[10. , 33. ],
       [ 9.5, 33. ],
       [ 9. , 33. ],
       [ 8.5, 33. ],
       [ 8. , 33. ],
       [ 7.5, 33. ],
       [ 7. , 33. ],
       [ 6.5, 33. ],
       [ 6. , 33. ],
       [ 5.5, 33. ],
       [ 5. , 33. ],
       [ 4.5, 33. ],
       [ 4. , 33. ],
       [ 4. , 33.5],
       [ 4. , 34. ],
       [ 4. , 34.5],
       [ 4. , 35. ],
       [ 4. , 35.5],
       [ 4. , 36. ],
       [ 4. , 36.5],
       [ 4. , 37. ],
       [ 4. , 37.5],
       [ 4. , 38. ],
       [ 4. , 38.5],
       [ 4. , 39. ],
       [ 4.5, 39. ],
       [ 5. , 39. ],
       [ 5.5, 39. ],
       [ 6. , 39. ],
       [ 6.5, 39. ],
       [ 7. , 39. ],
       [ 7.5, 39. ],
       [ 8. , 39. ],
       [ 8.5, 39. ],
       [ 9. , 39. ],
       [ 9.5, 39. ],
       [10. , 39. ],
       [10. , 38.5],
       [10. , 38. ],
       [10. , 37.5],
       [10. , 37. ],
       [10. , 36.5],
       [10. , 36. ],
       [10. , 35.5],
       [10. , 35. ],
       [10. , 34.5],
       [10. , 34. ],
       [10. ,

In [290]:
#define link lengths
l1 = 23.6# base link
l2 = 22 # tip link

#distance from origin to tip
r_squared = tip_path[:,0]**2 + tip_path[:,1]**2

#law of cos for angle between links
alpha_temp = (r_squared-l1**2-l2**2)/(-2*l1*l2)
print(alpha_temp)
alpha = np.arccos(alpha_temp)

print(alpha*rtd)

#vertical angle theorem for theta 2
theta2 = 180 - alpha*rtd

#triangle in link1 co-ordinant system for psi
psi = np.arctan2(l2*sind(theta2),l1+l2*cosd(theta2))*rtd

#angle of r to x-axis
beta = np.arctan2(tip_path[:,1],tip_path[:,0])*rtd

#difference in beta and psi is theta 1
theta1 = beta - psi

##theta2 = 180 - theta2
Check = 0
for i in range(len(theta1)):
    
    if theta2[i] < 0:
        theta2[i] = theta2[i]+360
for i in range(len(theta1)):
    if theta1[i] < 0 or theta1[i] >180:
        Check = 1
    if (theta2[i] > 90 and theta2[i] < 270) or (theta2[i] < 0):
        Check = 1
if Check == 0:
    print("Correct")
if Check == 1:
    print("Wrong")
print("\ntheta 1:\n",theta1)
print("theta 2:\n",theta2)

[-0.14256549 -0.13317604 -0.1242681  -0.11584168 -0.10789676 -0.10043336
 -0.09345146 -0.08695108 -0.0809322  -0.07539484 -0.07033898 -0.06576464
 -0.0616718  -0.09369222 -0.12619414 -0.15917758 -0.19264253 -0.22658898
 -0.26101695 -0.29592643 -0.33131741 -0.36718991 -0.40354391 -0.44037943
 -0.47769646 -0.48178929 -0.48636364 -0.49141949 -0.49695686 -0.50297573
 -0.50947612 -0.51645801 -0.52392142 -0.53186633 -0.54029276 -0.54920069
 -0.55859014 -0.52127311 -0.4844376  -0.44808359 -0.41221109 -0.37682011
 -0.34191063 -0.30748267 -0.27353621 -0.24007126 -0.20708783 -0.1745859
 -0.14256549 -0.14256549]
[ 98.19632714  97.65316232  97.13849173  96.6521742   96.19408739
  95.76412575  95.3621987   94.98822892  94.64215082  94.3239091
  94.03345751  93.77075761  93.53577777  95.37605374  97.24972136
  99.15916331 101.10703938 103.09633369 105.13041286 107.21309819
 109.34875651 111.54241565 113.79991342 116.12809276 118.5350617
 118.80232879 119.10185284 119.4339234  119.79887066 120.197068

In [291]:
theta_min = 0
theta_max = 180
min_new = 1000
max_new = 2000

myint = min_new + ((theta1-theta_min)*(max_new-min_new))/(theta_max-theta_min)
print(myint)


myint2 = min_new + (((180-(90+theta2))-theta_min)*(max_new-min_new))/(theta_max-theta_min)

#myint2_dif = 1500 - myint2
#myint2 = 1500 - myint2_dif -500
print('\n',myint2)
#print('\n',myint2_dif)

#theta2_new = theta_min + ((myint - min_new) * (theta_max - theta_min)) / (max_new - min_new)
#theta2_new

[1188.78214329 1191.80193613 1194.93258812 1198.17227879 1201.51916405
 1204.97137259 1208.5270031  1212.18412227 1215.94076358 1219.79492682
 1223.74457827 1227.78765163 1231.92204953 1237.26903471 1242.69609695
 1248.21007488 1253.81852798 1259.52986501 1265.35350169 1271.30005605
 1277.3815936  1283.61193856 1290.00707505 1296.58567184 1303.36978118
 1300.04404657 1296.81553874 1293.68628831 1290.65841311 1287.73412536
 1284.9157404  1282.20568728 1279.60652132 1277.12093914 1274.75179639
 1272.50212869 1270.37517649 1262.64548053 1255.16035248 1247.88843008
 1240.80372974 1233.88439606 1227.11179883 1220.46986582 1213.94458016
 1207.52359494 1201.195933   1194.95174974 1188.78214329 1188.78214329]

 [1045.5351508  1042.51756847 1039.65828739 1036.95652335 1034.41159663
 1032.02292086 1029.78999279 1027.71238289 1025.78972675 1024.02171723
 1022.40809727 1020.9486534  1019.64320983 1029.8669652  1040.27622978
 1050.88424062 1061.70577431 1072.75740939 1084.05784921 1095.6283233
 110

In [292]:
def break_into_two(breakint):
    MSB = breakint // 256
    LSB = breakint % 256
    return MSB, LSB

In [293]:
byte1 = np.zeros(len(theta1), dtype=int)
byte2 = np.zeros(len(theta1), dtype=int)
byte3 = np.zeros(len(theta1), dtype=int)
byte4 = np.zeros(len(theta1), dtype=int)

for i in range(len(theta1)):
    byte1[i], byte2[i] = break_into_two(myint[i])
    byte3[i], byte4[i] = break_into_two(myint2[i])

print(byte1,'\n\n',byte2,'\n\n\n',byte3,'\n\n',byte4)

[4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 5 5 5 5 5 5 5 5 5 5 5 4 4 4 4 4
 4 4 4 4 4 4 4 4 4 4 4 4 4] 

 [164 167 170 174 177 180 184 188 191 195 199 203 207 213 218 224 229 235
 241 247 253   3  10  16  23  20  16  13  10   7   4   2 255 253 250 248
 246 238 231 223 216 209 203 196 189 183 177 170 164 164] 


 [4 4 4 4 4 4 4 4 4 4 3 3 3 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4
 4 4 4 4 4 4 4 4 4 4 4 4 4] 

 [ 21  18  15  12  10   8   5   3   1   0 254 252 251   5  16  26  37  48
  60  71  83  95 108 121 134 136 137 139 141 143 146 148 151 154 157 161
 164 150 136 123 111  98  87  75  64  53  42  31  21  21]


In [294]:
# Send all path points to both servos
x = 1
for i in range(len(theta1)):
    WriteByte(ser, int(byte1[i]))   # servo 1 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte2[i]))   # servo 1 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte3[i]))   # servo 2 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte4[i]))   # servo 2 LSB
    time.sleep(0.05)

    # read back confirmation from Arduino
    #line1 = read_one_line(ser)  # servo 1 bytes echo
   # line2 = read_one_line(ser)  # servo 1 int echo
    #line3 = read_one_line(ser)  # servo 2 bytes echo
    #line4 = read_one_line(ser)  # servo 2 int echo
    #print(f"Step {i}: servo1={line2}  servo2={line4}")

   # print('\ntheta1:',theta1[i],'\ntheta2:',theta2[i],'\n','\n\n')
 
    if x == 1:
        x = 2
        
    else:
        
        while 1==1:
            response = ser.readline().decode('utf-8').strip()
            print(response)
            if response == "Ready":
                break
            else: 
                line1 = read_one_line(ser)  # servo 1 bytes echo
                line2 = read_one_line(ser)  # servo 1 int echo
                print(f"Step {i-1}: servo1={line1}  servo2={line2}")
                line3 = read_one_line(ser)  # servo 1 bytes echo
                line4 = read_one_line(ser)  # servo 1 int echo
                                              # servo 2 int echo
                print(f"Step {i}: servo1={line3}  servo2={line4}") 
            #response = ser.readline().decode('utf-8').strip()
            
                
        x = 1
    #time.sleep(1)  # pause between steps so servo has time to move

My int: 1188
Step 0: servo1=My int2: 1045  servo2=My int3: 1191
Step 1: servo1=My int4: 1042  servo2=
Ready

Step 2: servo1=My int: 1194  servo2=My int2: 1039
Step 3: servo1=My int3: 1198  servo2=My int4: 1036
Ready

Step 4: servo1=My int: 1201  servo2=My int2: 1034
Step 5: servo1=My int3: 1204  servo2=My int4: 1032
Ready

Step 6: servo1=My int: 1208  servo2=My int2: 1029
Step 7: servo1=My int3: 1212  servo2=My int4: 1027
Ready

Step 8: servo1=My int: 1215  servo2=My int2: 1025
Step 9: servo1=My int3: 1219  servo2=My int4: 1024
Ready

Step 10: servo1=My int: 1223  servo2=My int2: 1022
Step 11: servo1=My int3: 1227  servo2=My int4: 1020
Ready

Step 12: servo1=My int: 1231  servo2=My int2: 1019
Step 13: servo1=My int3: 1237  servo2=My int4: 1029
Ready

Step 14: servo1=My int: 1242  servo2=My int2: 1040
Step 15: servo1=My int3: 1248  servo2=My int4: 1050
Ready

Step 16: servo1=My int: 1253  servo2=My int2: 1061
Step 17: servo1=My int3: 1259  servo2=My int4: 1072
Ready

Step 18: servo1=My 

In [124]:
#byte3, byte4 = break_into_two(1200)

In [125]:
#WriteByte(ser, MSB)
#WriteByte(ser, LSB)

In [126]:

WriteByte(ser,byte1)#<--
time.sleep(0.1)

WriteByte(ser,byte2)#<--
time.sleep(0.1)

#WriteByte(ser,byte3)#<--
#time.sleep(0.1)

#WriteByte(ser,byte4)#<--
#time.sleep(0.1)
next_line = read_one_line(ser)
extra = read_all(ser)
print('next_line: %s' % next_line)
print('extra: %s' % extra)

TypeError: only length-1 arrays can be converted to Python scalars

In [ ]:
byte3, byte4 = break_into_two(1500)

In [ ]:
WriteByte(ser,byte3)#<--
time.sleep(0.1)

WriteByte(ser,byte4)#<--
time.sleep(0.1)
#WriteByte(ser,byte3)#<--
#time.sleep(0.1)

#WriteByte(ser,byte4)#<--
#time.sleep(0.1)
next_line = read_one_line(ser)
extra = read_all(ser)
print('next_line: %s' % next_line)
print('extra: %s' % extra)

In [286]:
#ser.close()

In [ ]:
for i in range(1100, 1900, 50):
    byte3, byte4 = break_into_two(i)
    WriteByte(ser,byte3)#<--
    time.sleep(0.1)

    WriteByte(ser,byte4)#<--
    time.sleep(0.1)
    #WriteByte(ser,byte3)#<--
    #time.sleep(0.1)

    #WriteByte(ser,byte4)#<--
    #time.sleep(0.1)
    next_line = read_one_line(ser)
    extra = read_all(ser)
    print('next_line: %s' % next_line)
    print('extra: %s' % extra)
    time.sleep(0.5)
    print(i) 

- How do we break this into two bytes?
- How do we find the most significant byte?
- How do we find the least significant byte?

In [ ]:
ser.close()